# w9_flash_byol_vicreg.ipynb — negative-free control blitz (BYOL + VICReg)

Dedicated pod for the negative-free control block that pairs with the
30-cell CE/I grid. New cells: **epdg_*** = canonical VICReg (all three terms
on the expander-output pair) with the expander HARMONIZED to the grid's exp
module (128→256→512, no LayerNorm) — so `expce vs epdg` isolates the loss
functional (CE vs V/C+I) inside an identical room, at identical anchors and
protocol. Already-finished family rows (byol, byol2, vic, vic2, epd, epdb)
are listed for the readout and skip automatically. Claims are heartbeat-
compatible with the campaign pods. AUTO-STOPS when drained.

Note: anchors never enter VICReg/BYOL TRAINING (negative-free) — anchor cap
is eval-side only for this family; the comparison to the grid is family-
level + the harmonized same-E pair.


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

BV = [
    # NEW: grid-harmonized canonical VICReg (three weight points)
    ("wcle_epdg_v20i10c15_cetf", 512),   # our best allocation first
    ("wcle_epdg_v25i25c1_cetf", 512),    # image-paper weights
    # family rows (done -> skip; kept so the readout prints the full block)
    ("wcle_byol_bytf", 512),
    ("wcle_byol2_bytf", 512),
    ("wcle_vic_cetf", 512),
    ("wcle_vic2_cetf", 512),
    ("wcle_epd_v25i25c1_cetf", 512),
    ("wcle_epd_v20i10c20_cetf", 512),
    ("wcle_epd_v20i10c15_cetf", 512),
    ("wcle_epdb_v25i25c1_cetf", 512),
    ("wcle_epdb_v20i10c20_cetf", 512),
    ("wcle_epdb_v20i10c15_cetf", 512),
]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(BV)} rows ({sum(1 for _ in BV)} listed; done ones skip)")


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Drain across ALL GPUs (heartbeat claims; done rows skip).
import os, queue, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
jobs = queue.Queue()
for arm, cap in BV:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / J.result_name(nm)).exists():
        print(f"[skip] {nm} done"); continue
    jobs.put((arm, cap, nm))
fails = []

def worker(gpu):
    while True:
        try:
            arm, cap, nm = jobs.get_nowait()
        except queue.Empty:
            return
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True)
            continue
        log = logd / f"{arm}_g{cap}.log"
        cmd = ["python", "-u", J.FS_WORKER,
               "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(J.FS_EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpu}] start {nm}", flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))
        if p.returncode != 0:
            fails.append((nm, str(log)))
        print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
              + f" {nm} [{(time.time()-t0)/60:.1f} min]", flush=True)

stop_evt = threading.Event()
mon = threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True)
mon.start()
gpus = J.detect_gpus()
threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
stop_evt.set()
print(f"BV blitz drained in {(time.time()-t0)/3600:.1f} h; {len(fails)} failed")
for nm, lg in fails:
    print("  FAILED:", nm, "->", lg)


In [ ]:
# Readout: negative-free block vs the grid's same-E CE reference.
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = ([("wcle_expce_cetf", "expce (grid CE@E ref)")]
        + [(a, a.replace("wcle_", "").replace("_cetf", "").replace("_bytf", ""))
           for a, _ in BV])
for arm, lab in ROWS:
    p = Path(OUT_DIR) / f"ft4var_w9_{arm}_fp_best.json"
    if not p.exists():
        print(f"{lab:28s} (missing)"); continue
    d = json.loads(p.read_text())
    runs = d["per_seed"]
    r = {v: np.mean([x[v]["h1"] for x in runs]) for v in VORD}
    print(f"{lab:28s} ep{d.get('best_ep'):>4} "
          + " ".join(f"{v[:3]}:{r[v]:.3f}" for v in VORD)
          + f" m4:{np.mean(list(r.values())):.3f} "
          f"tag:{np.mean([x['noname']['tag'] for x in runs]):.3f}")


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)